# M7 — Eksploracja prawdziwego datasetu LeRobot

**Kurs Unitree G1 · Modul 7 · wariant A (przegladarka)**

Nie masz srodowiska do teleoperacji? Nie szkodzi — w tym notebooku rozbierzesz
na czesci PRAWDZIWY dataset demonstracji w formacie LeRobot z HuggingFace:
dokladnie taki, jaki sam wyprodukujesz w wariancie B (i jaki konsumuje
fine-tuning GR00T w module 8).

Czego sie nauczysz:
1. Struktura LeRobot: parquet (dane) + MP4 (wideo) + meta (JSON)
2. Co siedzi w pojedynczym epizodzie (stany, akcje, timestampy)
3. Ocena jakosci demonstracji wg kryteriow z sekcji 7.3


In [ ]:
# Instalacja (Colab: ~1 min)
%pip -q install datasets huggingface_hub pandas pyarrow matplotlib pillow

In [ ]:
# Publiczny dataset LeRobot: ALOHA robi kawe (statyczny manipulator dwureki).
# Format identyczny jak dla G1 — rozni sie tylko liczba stawow i kamerami.
from huggingface_hub import HfApi, hf_hub_download
import json

REPO = "lerobot/aloha_static_coffee"
api = HfApi()
info = api.dataset_info(REPO)
print(f"Dataset: {REPO}")
print(f"Pliki ({len(list(info.siblings))}):")
for s in list(info.siblings)[:15]:
    print("  ", s.rfilename)
print("  ...")

In [ ]:
# Metadane: info.json mowi wszystko o strukturze
path = hf_hub_download(REPO, "meta/info.json", repo_type="dataset")
meta = json.load(open(path))
print(json.dumps({k: meta[k] for k in ["total_episodes", "total_frames", "fps", "robot_type"] if k in meta}, indent=2))
print("\nFeatures (obserwacje + akcje):")
for name, spec in meta.get("features", {}).items():
    print(f"  {name}: shape={spec.get('shape')} dtype={spec.get('dtype')}")

**Zatrzymaj sie:** porownaj z sekcja 7.4 kursu. Widzisz `observation.state`
(pozycje stawow), `action` (cele pozycji) i strumienie kamer? To dokladnie ten
uklad, ktory dla G1 generuje `convert_hdf5_to_lerobot.py`.

In [ ]:
# Wczytaj jeden epizod danych (parquet) i obejrzyj sygnaly
import pandas as pd
from huggingface_hub import list_repo_files

files = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".parquet")]
print("przykladowy plik:", files[0])
p = hf_hub_download(REPO, files[0], repo_type="dataset")
df = pd.read_parquet(p)
print(df.shape)
df.head(3)

In [ ]:
# Wykres: trajektorie akcji w czasie — "ksztalt" demonstracji
import numpy as np
import matplotlib.pyplot as plt

ep = df[df["episode_index"] == df["episode_index"].iloc[0]] if "episode_index" in df else df
actions = np.stack(ep["action"].to_numpy())
t = np.arange(len(actions)) / meta.get("fps", 50)

plt.figure(figsize=(10, 4))
for j in range(min(6, actions.shape[1])):
    plt.plot(t, actions[:, j], label=f"staw {j}")
plt.xlabel("czas [s]"); plt.ylabel("akcja (pozycja docelowa)")
plt.title("Pierwsze 6 wymiarow akcji w epizodzie")
plt.legend(fontsize=7); plt.grid(alpha=0.3); plt.show()

# PYTANIE: czy ruchy sa plynne (kryterium 7.3), czy widzisz szarpniecia?

In [ ]:
# Ocena jakosci wg 7.3: plynnosc = norma drugiej roznicy (jerk proxy)
jerk = np.abs(np.diff(actions, n=2, axis=0)).mean(axis=1)
print(f"sredni |jerk| epizodu: {jerk.mean():.5f}")
print(f"max |jerk|: {jerk.max():.5f} w klatce {jerk.argmax()}")

plt.figure(figsize=(10, 2.5))
plt.plot(t[2:], jerk)
plt.xlabel("czas [s]"); plt.ylabel("|jerk| proxy")
plt.title("Gladkosc demonstracji — piki = szarpniecia (kandydat do odrzucenia?)")
plt.grid(alpha=0.3); plt.show()

## Zadanie koncowe (15-20 min)

1. Policz `total_episodes` i sredni czas trwania epizodu (frames/fps).
2. Wybierz 3 epizody i porownaj ich profile jerku — ktory odrzucilbys wg 7.3 i czemu?
3. Otworz strone datasetu na HF (zakladka **Dataset Viewer**) i obejrzyj wideo
   najlepszego i najgorszego epizodu — zgadza sie z Twoja ocena z danych?
4. W 3 zdaniach: czym rozniloby sie to dla G1 (podpowiedz: liczba stawow,
   kamery ego/zewnetrzne, locomotion w tle).

Wnioski zapisz — omowisz je na sesji indywidualnego supportu.

In [ ]:
# Miejsce na Twoje rozwiazanie zadania 1-2:
n_eps = meta.get("total_episodes")
print(f"epizodow: {n_eps}, sredni czas: {meta.get('total_frames', 0) / max(n_eps,1) / meta.get('fps', 50):.1f} s")
# ... Twoj kod ...